In [1]:
import sqlite3
import pandas as pd
import os

In [2]:
import kagglehub
import pandas as pd
# Download latest version
path = kagglehub.dataset_download("parsakh/global-e-commerce-and-supply-chain-database")

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sqlite3

conn = sqlite3.connect("ecommerce_supply_chain.db")
cursor = conn.cursor()

In [4]:
data_dir = "/home/codespace/.cache/kagglehub/datasets/parsakh/global-e-commerce-and-supply-chain-database/versions/1"

tables = {
    "customers": "customers.csv",
    "products": "products.csv",
    "suppliers": "supplier_costs.csv",
    "transactions": "transactions.csv",
    "inventory": "inventory.csv",
}

for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(data_dir, filename))
    df.to_sql(table_name, conn, if_exists="append", index=False)
    print(f"Loaded {table_name}: {df.shape[0]} rows")

Loaded customers: 8000 rows
Loaded products: 500 rows
Loaded suppliers: 998 rows
Loaded transactions: 100000 rows
Loaded inventory: 500 rows


In [5]:
conn = sqlite3.connect('ecommerce_supply_chain.db')

sql_query = """SELECT *
FROM transactions t
"""
transactions_df = pd.read_sql_query(sql_query, conn)

conn.close()

In [6]:
transactions_df.head()

,transaction_id,customer_id,product_id,date,quantity,unit_price_usd,discount_pct,revenue_usd,cost_usd,profit_usd,shipping_cost_usd,channel,payment_method,status,country,category
0,TXN000001,CUST05879,PROD0230,2023-05-15,1,21.66,0.0,21.66,13.00,8.66,7.24,organic_search,paypal,completed,UK,Home & Kitchen
1,TXN000002,CUST00461,PROD0055,2022-08-21,1,188.93,0.0,188.93,85.02,103.91,13.66,social_media,credit_card,completed,USA,Clothing
2,TXN000003,CUST03473,PROD0353,2023-08-28,5,202.21,0.0,1011.05,606.65,404.40,108.68,direct,paypal,completed,France,Home & Kitchen
3,TXN000004,CUST07458,PROD0253,2024-05-31,4,249.76,0.2,799.23,549.48,249.75,0.00,email,bank_transfer,completed,UK,Sports & Outdoors
4,TXN000005,CUST06099,PROD0206,2023-04-22,2,287.63,0.0,575.26,258.86,316.40,9.30,organic_search,credit_card,completed,USA,Clothing


In [7]:
conn = sqlite3.connect('ecommerce_supply_chain.db')

sql_query = """SELECT *
FROM customers c
"""
customers_df = pd.read_sql_query(sql_query, conn)

conn.close()

In [8]:
customers_df.head()

,customer_id,first_name,last_name,country,currency,age,gender,registration_date,is_premium,email_verified,email
0,CUST00001,Laura,Brown,UK,GBP,21,F,2020-10-12,0,1,laura.brown0001@email.com
1,CUST00002,Claire,Brown,Australia,AUD,47,M,2020-07-13,1,1,claire.brown0002@email.com
2,CUST00003,Thomas,Allen,Canada,CAD,68,M,2020-06-04,0,1,thomas.allen0003@email.com
3,CUST00004,William,Jackson,France,EUR,59,M,2020-12-12,0,0,william.jackson0004@email.com
4,CUST00005,Nina,Mueller,USA,USD,56,F,2021-10-24,0,1,nina.mueller0005@email.com


TOP Buying Costumers by Total Purchase Amount

In [114]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('ecommerce_supply_chain.db')

sql_query = """SELECT  
  case
   when c.age <18 then 'Minor'
   when c.age between 18 and 35 then 'Young Adult'
   when c.age between 36 and 55 then 'Adult'
   when c.age > 55 then 'Elderly'
 END as age_range, 

 AVG(t.revenue_usd) AS average_purchase_amount ,p.category

 FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
JOIN products p ON t.product_id = p.product_id
GROUP BY age_range
ORDER BY average_purchase_amount DESC;"""

top_customers_by_age_df = pd.read_sql_query(sql_query, conn)

conn.close()

In [115]:
top_customers_by_age_df.head()

,age_range,average_purchase_amount,category
0,Elderly,534.043215,Beauty & Health
1,Young Adult,531.991876,Toys & Games
2,Adult,520.717992,Home & Kitchen


In [141]:
conn = sqlite3.connect('ecommerce_supply_chain.db')

sql_query = """SELECT  
  CASE
  when c.is_premium = 1 THEN 'Premium'
  ELSE 'Regular'
  END as customer_type,
 AVG(t.revenue_usd) AS average_purchase_amount,COUNT(DISTINCT c.customer_id) AS customer_count

 FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
JOIN products p ON t.product_id = p.product_id
GROUP BY customer_type
ORDER BY average_purchase_amount DESC;"""

premium_customers_df = pd.read_sql_query(sql_query, conn)

conn.close()

In [142]:
premium_customers_df.head()

,customer_type,average_purchase_amount,customer_count
0,Regular,535.801287,6237
1,Premium,505.135342,1740


In [151]:
conn = sqlite3.connect('ecommerce_supply_chain.db')

sql_query = """SELECT  
  case
   when c.age <18 then 'Minor'
   when c.age between 18 and 35 then 'Young Adult'
   when c.age between 36 and 55 then 'Adult'
   when c.age > 55 then 'Elderly'
 END as age_range, 
  CASE
  when c.is_premium = 1 THEN 'Premium'
  ELSE 'Regular'
  END as customer_type,
 AVG(t.revenue_usd) AS average_purchase_amount,COUNT(DISTINCT c.customer_id) AS customer_count

 FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
JOIN products p ON t.product_id = p.product_id
GROUP BY age_range,customer_type
ORDER BY age_range,customer_count DESC;"""

premium_customers_df = pd.read_sql_query(sql_query, conn)

conn.close()

In [152]:
premium_customers_df.head(10)

,age_range,customer_type,average_purchase_amount,customer_count
0,Adult,Regular,528.400516,2209
1,Adult,Premium,498.595041,614
2,Elderly,Regular,536.255886,2084
3,Elderly,Premium,525.681901,571
4,Young Adult,Regular,543.148155,1944
5,Young Adult,Premium,493.336997,555
